<img width="8%" alt="Archivist" src="https://upload.wikimedia.org/wikipedia/commons/thumb/e/e1/Biblioth%C3%A8que_Inguimbertine_salle_X.jpg/1920px-Biblioth%C3%A8que_Inguimbertine_salle_X.jpg" style="border-radius: 15%">

# Archivist - Indexing
<a href="https://github.com/mbasri/archivist">Give Feedback</a> | <a href="https://github.com/mbasri/archivist">Bug report</a>

**Tags:** #archivist #scan #embed #index #qdrant #ollama #llamaindex #rag #python

**Author:** [Mohamed BASRI](https://github.com/mbasri)

**Last update:** 2026-04-04 (Created: 2026-04-02)

**Description:** Full indexing pipeline — reads extracted text files from `extracted/` (output of `01-extract.ipynb`), skips already indexed files (hash-based via `extract_index.json`), chunks text content, embeds each chunk with LlamaIndex + Ollama, and stores vectors into Qdrant. Swap the embedding model or vector store in a single line.

**References:**
- [LlamaIndex](https://docs.llamaindex.ai/)
- [LlamaIndex Qdrant integration](https://docs.llamaindex.ai/en/stable/examples/vector_stores/qdrant/)
- [LlamaIndex Ollama embedding](https://docs.llamaindex.ai/en/stable/examples/embeddings/ollama_embedding/)
- [nomic-embed-text](https://ollama.com/library/nomic-embed-text)
- [Archivist project](https://github.com/mbasri/archivist)

## Setup

### Install dependencies

In [ ]:
import sys
import subprocess
from pathlib import Path

project_root = Path().resolve()
project_root = project_root.parent if project_root.name == "notebooks" else project_root

result = subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "-r", str(project_root / "requirements.txt")],
    capture_output=True, text=True
)

print(result.stdout[-3000:] if result.stdout else "")
print(result.stderr[-3000:] if result.stderr else "")

if result.returncode != 0:
    print(f"✗ Install failed (exit code {result.returncode})")
else:
    print("✓ Dependencies installed")

## Input

### Import libraries

In [ ]:
import hashlib
import json
import os
from collections import defaultdict
from dotenv import load_dotenv

from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

load_dotenv(project_root / ".env", override=True)
print("✓ .env loaded")

### Setup Variables
- `nas_path`: path to the directory to scan and index
- `text_extensions`: files read directly as plain text
- `docling_extensions`: files converted to text via Docling (PDF, Office, images)
- `embed_model`: LlamaIndex embedding model — swap to `OpenAIEmbedding` or `BedrockEmbedding` for cloud
- `chunk_size` / `chunk_overlap`: chunking parameters (in tokens)
- `collection_name`: Qdrant collection name
- `qdrant_host` / `qdrant_port`: Qdrant connection
- `ollama_base_url`: Ollama connection

In [ ]:
extracted_dir      = project_root / os.getenv("EXTRACTED_DIR", "extracted")
extract_index_path = project_root / os.getenv("EXTRACT_INDEX", "extract_index.json")
index_path         = project_root / os.getenv("INDEX_FILE",    "index.json")

chunk_size      = int(os.getenv("CHUNK_SIZE",    512))
chunk_overlap   = int(os.getenv("CHUNK_OVERLAP", 50))

collection_name = os.getenv("COLLECTION_NAME", "archivist")
qdrant_host     = os.getenv("QDRANT_HOST",     "localhost")
qdrant_port     = int(os.getenv("QDRANT_PORT", 6333))
ollama_base_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
embedding_model = os.getenv("EMBEDDING_MODEL", "nomic-embed-text")

embed_model = OllamaEmbedding(model_name=embedding_model, base_url=ollama_base_url)

print(f"Extracted dir : {extracted_dir}")
print(f"Qdrant        : {qdrant_host}:{qdrant_port}")
print(f"Ollama        : {ollama_base_url}")
print(f"Embedding     : {embedding_model}")
print(f"Index file    : {index_path}")

## Model

### Load index

In [ ]:
if index_path.exists():
    with open(index_path, "r") as f:
        index = json.load(f)
else:
    index = {}

print(f"Index loaded: {len(index)} file(s) already indexed")

### Load extracted files and filter already indexed

In [ ]:
if extract_index_path.exists():
    with open(extract_index_path) as f:
        extract_index = json.load(f)
else:
    extract_index = {}

to_index = []
for h, meta in extract_index.items():
    if h not in index:
        to_index.append((Path(meta["extracted"]), h))

print(f"Extracted files total : {len(extract_index)}")
print(f"Already indexed       : {len(index)}")
print(f"To index              : {len(to_index)}")

### Initialize Qdrant vector store

In [ ]:
qdrant_client = QdrantClient(host=qdrant_host, port=qdrant_port)

# Swap QdrantVectorStore for ChromaVectorStore or PGVectorStore to change vector DB
vector_store    = QdrantVectorStore(client=qdrant_client, collection_name=collection_name)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

print(f"Connected to Qdrant — collection: '{collection_name}'")

### Chunk, embed, and index
LlamaIndex handles chunking and embedding. Each extracted file is processed independently so `index.json` is updated after every successful file.

In [ ]:
splitter        = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
indexed, errors = 0, []
total           = len(to_index)

for i, (file, h) in enumerate(to_index, 1):
    try:
        documents = SimpleDirectoryReader(input_files=[str(file)]).load_data()

        VectorStoreIndex.from_documents(
            documents,
            storage_context=storage_context,
            embed_model=embed_model,
            transformations=[splitter],
            show_progress=False,
        )

        index[h] = str(file)
        with open(index_path, "w") as f:
            json.dump(index, f, indent=2)

        print(f"  [{i}/{total}] [OK]  {file.name}")
        indexed += 1

    except Exception as e:
        print(f"  [{i}/{total}] [ERR] {file.name} → {e}")
        errors.append((file, str(e)))

## Output

### Display result

In [ ]:
info = qdrant_client.get_collection(collection_name)

print(f"Files indexed           : {indexed}")
print(f"Errors                  : {len(errors)}")
print(f"Total vectors in Qdrant : {info.points_count}")

if errors:
    print("\nFailed files:")
    for file, err in errors:
        print(f"  - {file.name}: {err}")